# Machine Learning model on Premier League 2023/24

## Using logistic regression and random forest with fine-tuned hyperparameter (GridSearch)*

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print('Success')

Success


In [2]:
# importing dataframe
url = "https://www.football-data.co.uk/mmz4281/2324/E0.csv" 
df = pd.read_csv(url)
# Choosing which columns to keep
columns_to_keep = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', # Goals and Result
    'HS', 'AS',            # Home Shots Away Shots (leading indicator)
    'HST', 'AST',          # Home Shots on Target, Away Shots on Target 
    'HC', 'AC',            # Home Corners, Away Corners 
    'B365H', 'B365D', 'B365A' # Odds
]
odds_df = df[['B365H', 'B365D', 'B365A']]
df = df[columns_to_keep]
#Date object + reorder chronologically
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values(by='Date').reset_index(drop=True)


In [3]:
# Calculate match points(Win=3, Draw=1, Loss=0)
def calc_points(gf, ga):
    return np.where(gf > ga, 3, np.where(gf == ga, 1, 0))

In [4]:
# isolate home games
home_df = df[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']].copy()
home_df.columns = ['Date', 'Team', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'SOTFor', 'SOTAgainst', 'CornersFor', 'CornersAgainst']
home_df['MatchPoints'] = calc_points(home_df['GoalsFor'], home_df['GoalsAgainst'])


In [5]:
# isolate away games
away_df = df[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC']].copy()
away_df.columns = ['Date', 'Team', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'SOTFor', 'SOTAgainst', 'CornersFor', 'CornersAgainst']
away_df['MatchPoints'] = calc_points(away_df['GoalsFor'], away_df['GoalsAgainst'])


In [6]:
# combine into one object
master_timeline = pd.concat([home_df, away_df]).sort_values(by=['Team', 'Date']).reset_index(drop=True)


In [7]:
grouped = master_timeline.groupby('Team')
master_timeline['Rolling_5_Points'] = grouped['MatchPoints'].transform(lambda x: x.shift(1).rolling(5).sum())

In [8]:
for stat in ['GoalsFor', 'ShotsFor', 'SOTFor', 'CornersFor', 'GoalsAgainst', 'ShotsAgainst', 'SOTAgainst', 'CornersAgainst']:
    new_col = f'EMA_5_{stat}'
    master_timeline[new_col] = grouped[stat].transform(lambda x: x.shift(1).ewm(span=5, adjust=False).mean())


In [9]:
# 10-Game simple moving average
for stat in ['GoalsFor', 'ShotsFor', 'SOTFor', 'CornersFor', 'GoalsAgainst', 'ShotsAgainst', 'SOTAgainst', 'CornersAgainst']:
    new_col = f'SMA_10_{stat}'
    master_timeline[new_col] = grouped[stat].transform(lambda x: x.shift(1).rolling(window=10).mean())

In [10]:
features_to_keep = ['Date', 'Team', 'Rolling_5_Points'] + [col for col in master_timeline.columns if 'EMA_5' in col or 'SMA_10' in col]
calc_stats = master_timeline[features_to_keep]


In [11]:
df = pd.merge(df, calc_stats, left_on=['Date', 'HomeTeam'], right_on=['Date', 'Team'], how='left')
# Add a "Home_" prefix to all the calculated columns
df = df.rename(columns={col: f'Home_{col}' for col in features_to_keep if col not in ['Date', 'Team']})
df = df.drop(columns=['Team'])

# Attach stats to the Away Team
df = pd.merge(df, calc_stats, left_on=['Date', 'AwayTeam'], right_on=['Date', 'Team'], how='left')
# Add an "Away_" prefix to all the calculated columns
df = df.rename(columns={col: f'Away_{col}' for col in features_to_keep if col not in ['Date', 'Team']})
df = df.drop(columns=['Team'])

# Clean out the un-modelable first 10 weeks of the season
df = df.dropna().reset_index(drop=True)

In [12]:
df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A,Home_Rolling_5_Points,Home_EMA_5_GoalsFor,Home_EMA_5_ShotsFor,Home_EMA_5_SOTFor,Home_EMA_5_CornersFor,Home_EMA_5_GoalsAgainst,Home_EMA_5_ShotsAgainst,Home_EMA_5_SOTAgainst,Home_EMA_5_CornersAgainst,Home_SMA_10_GoalsFor,Home_SMA_10_ShotsFor,Home_SMA_10_SOTFor,Home_SMA_10_CornersFor,Home_SMA_10_GoalsAgainst,Home_SMA_10_ShotsAgainst,Home_SMA_10_SOTAgainst,Home_SMA_10_CornersAgainst,Away_Rolling_5_Points,Away_EMA_5_GoalsFor,Away_EMA_5_ShotsFor,Away_EMA_5_SOTFor,Away_EMA_5_CornersFor,Away_EMA_5_GoalsAgainst,Away_EMA_5_ShotsAgainst,Away_EMA_5_SOTAgainst,Away_EMA_5_CornersAgainst,Away_SMA_10_GoalsFor,Away_SMA_10_ShotsFor,Away_SMA_10_SOTFor,Away_SMA_10_CornersFor,Away_SMA_10_GoalsAgainst,Away_SMA_10_ShotsAgainst,Away_SMA_10_SOTAgainst,Away_SMA_10_CornersAgainst
0,2023-11-04,Fulham,Man United,0,1,A,18,12,3,5,9,4,3.20,3.60,2.2,5.0,0.915968,11.431083,4.189148,5.003556,1.347813,13.219530,5.120205,4.574963,0.9,10.6,3.5,5.0,1.6,12.5,5.6,4.7,9.0,0.964284,13.091043,4.520500,6.515826,1.753645,14.807956,6.381649,7.540212,1.1,15.1,4.8,6.9,1.6,14.0,5.4,6.3
1,2023-11-04,Brentford,West Ham,3,2,H,16,12,4,2,4,3,2.05,3.60,3.5,7.0,1.815170,13.480719,5.531118,3.914596,0.766550,13.526038,3.081644,6.591475,1.6,14.3,4.9,4.7,1.2,13.2,3.8,5.4,4.0,1.007875,11.884570,3.586344,5.008484,1.935528,13.991160,5.056648,5.385104,1.6,11.7,4.0,4.7,1.7,17.4,5.8,8.0
2,2023-11-04,Burnley,Crystal Palace,0,2,A,17,4,5,3,12,1,2.75,3.25,2.6,3.0,0.760301,8.128944,2.586852,5.461718,2.444140,15.805467,6.559874,6.561398,0.8,9.7,2.7,5.6,2.5,16.1,6.4,5.5,5.0,0.609308,12.377940,3.143271,6.946451,1.778286,12.138698,3.774983,4.780674,0.8,12.8,4.2,4.7,1.3,12.7,3.6,5.4
3,2023-11-04,Everton,Brighton,1,1,D,10,7,4,2,3,3,3.10,3.60,2.2,9.0,1.132602,13.550475,4.245796,4.327948,0.907788,14.655286,3.513590,6.490474,1.0,14.9,4.9,5.2,1.4,12.9,3.9,5.6,5.0,1.543210,13.403648,5.142153,5.239598,1.903216,11.850531,5.180562,2.718844,2.3,15.4,6.4,5.6,1.9,12.5,4.5,3.8
4,2023-11-04,Man City,Bournemouth,6,1,H,21,5,8,1,12,1,1.10,11.00,19.0,9.0,2.082965,15.234720,7.108774,7.027790,0.660570,6.980084,2.586039,4.411065,2.2,16.2,7.0,6.5,0.7,6.8,2.3,3.6,3.0,1.052279,10.768430,4.419143,6.398618,1.977392,14.800691,5.587614,7.101204,0.8,11.4,4.1,5.1,2.1,17.3,6.2,6.3


In [13]:
target_map = {'A': 0, 'D': 1, 'H': 2}
y = df['FTR'].map(target_map)

In [14]:
columns_to_drop = [
    'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', 
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'B365H', 'B365D', 'B365A'
]
X = df.drop(columns=columns_to_drop)

In [15]:
split_index = int(len(df) * 0.8)

In [16]:
X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

In [17]:
X_test = X.iloc[split_index:].copy()
y_test = y.iloc[split_index:].copy()

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) #learn parameters and the code it
X_test_scaled = scaler.transform(X_test)
X_test_scaled = scaler.transform(X_test)
print(f"Training Set: {X_train_scaled.shape[0]} matches")
print(f"Testing Set: {X_test_scaled.shape[0]} matches")
print(f"Features per match: {X_train_scaled.shape[1]}")

Training Set: 224 matches
Testing Set: 56 matches
Features per match: 34


In [19]:
# max_iter=1000 to ensure enough cycles to find optimal weights.
lr_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=45)
lr_model.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",45
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [20]:
y_pred = lr_model.predict(X_test_scaled)
y_proba = lr_model.predict_proba(X_test_scaled)
print(y_pred[:5])
print(y_proba[:5])

[1 1 2 0 0]
[[0.23982106 0.39104112 0.36913783]
 [0.20787347 0.64280376 0.14932277]
 [0.27504383 0.2233829  0.50157327]
 [0.47039661 0.12557585 0.40402754]
 [0.49989303 0.27567685 0.22443012]]


In [21]:
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_proba)

In [22]:
print("--- LOGISTIC REGRESSION BASELINE ---")
print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Log Loss:  {loss:.4f}")
print("-" * 36)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)'], zero_division=0))

--- LOGISTIC REGRESSION BASELINE ---
Accuracy:  37.50%
Log Loss:  1.2213
------------------------------------

Classification Report:

              precision    recall  f1-score   support

Away Win (0)       0.53      0.43      0.47        21
    Draw (1)       0.00      0.00      0.00         8
Home Win (2)       0.63      0.44      0.52        27

    accuracy                           0.38        56
   macro avg       0.39      0.29      0.33        56
weighted avg       0.50      0.38      0.43        56



## Clearly, large log-loss and low precision. Not an appropriate model. For example, only 21/56 matches guessed correctly- better off just guessing home every time!
## 0 draws guessed correctly- model weights winning too high and can't price draws well.

In [23]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_proba_rf = rf_model.predict_proba(X_test_scaled)


In [24]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
loss_rf = log_loss(y_test, y_proba_rf)

print("--- RANDOM FOREST CLASSIFIER ---")
print(f"Accuracy:  {accuracy_rf * 100:.2f}%")
print(f"Log Loss:  {loss_rf:.4f}")
print("-" * 32)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)'], zero_division=0))

--- RANDOM FOREST CLASSIFIER ---
Accuracy:  53.57%
Log Loss:  0.9715
--------------------------------

Classification Report:

              precision    recall  f1-score   support

Away Win (0)       0.60      0.57      0.59        21
    Draw (1)       0.14      0.25      0.18         8
Home Win (2)       0.73      0.59      0.65        27

    accuracy                           0.54        56
   macro avg       0.49      0.47      0.47        56
weighted avg       0.60      0.54      0.56        56



In [25]:
results_df = df.iloc[split_index:][['Date', 'HomeTeam', 'AwayTeam', 'FTR']].copy()

# Add the model's  prediction and the actual outcome
results_df['Predicted_Result'] = y_pred_rf
results_df['Actual_Result'] = y_test.values

# Map the numbers back to letters
inverse_target_map = {0: 'A', 1: 'D', 2: 'H'}
results_df['Predicted_Result'] = results_df['Predicted_Result'].map(inverse_target_map)

# Add the exact probability confidence for each outcome
results_df['Prob_Away(%)'] = (y_proba_rf[:, 0] * 100).round(2)
results_df['Prob_Draw(%)'] = (y_proba_rf[:, 1] * 100).round(2)
results_df['Prob_Home(%)'] = (y_proba_rf[:, 2] * 100).round(2)

# Display the first 10 matches
display(results_df.head(10))

,Date,HomeTeam,AwayTeam,FTR,Predicted_Result,Actual_Result,Prob_Away(%),Prob_Draw(%),Prob_Home(%)
224,2024-04-20,Sheffield United,Burnley,A,H,0,30.5,34.5,35.0
225,2024-04-20,Wolves,Arsenal,A,A,0,71.5,15.5,13.0
226,2024-04-21,Crystal Palace,West Ham,H,A,2,51.5,26.0,22.5
227,2024-04-21,Everton,Nott'm Forest,H,A,2,40.0,26.0,34.0
228,2024-04-21,Aston Villa,Bournemouth,H,A,2,42.0,35.5,22.5
229,2024-04-21,Fulham,Liverpool,A,A,0,46.5,34.5,19.0
230,2024-04-23,Arsenal,Chelsea,H,H,2,23.0,34.0,43.0
231,2024-04-24,Wolves,Bournemouth,A,D,0,40.0,45.0,15.0
232,2024-04-24,Crystal Palace,Newcastle,H,A,2,40.0,29.5,30.5
233,2024-04-24,Everton,Liverpool,H,A,2,61.0,22.5,16.5


In [26]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_estimators': [100, 200, 300],  'max_depth': [None, 5, 10, 15], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]
}

In [27]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=param_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1, 
    verbose=1  
)

In [28]:
grid_search.fit(X_train_scaled, y_train)
best_rf_model = grid_search.best_estimator_

print("\n--- OPTIMAL HYPERPARAMETERS FOUND ---")
print(grid_search.best_params_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits

--- OPTIMAL HYPERPARAMETERS FOUND ---
{'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}


In [29]:
y_pred_opt = best_rf_model.predict(X_test_scaled)
y_proba_opt = best_rf_model.predict_proba(X_test_scaled)

accuracy_opt = accuracy_score(y_test, y_pred_opt)
loss_opt = log_loss(y_test, y_proba_opt)

In [30]:
print("\n--- OPTIMIZED RANDOM FOREST RESULTS ---")
print(f"Accuracy:  {accuracy_opt * 100:.2f}%")
print(f"Log Loss:  {loss_opt:.4f}")
print("-" * 32)


--- OPTIMIZED RANDOM FOREST RESULTS ---
Accuracy:  57.14%
Log Loss:  0.9752
--------------------------------


In [31]:
market_df = df.iloc[split_index:][['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'B365H', 'B365D', 'B365A']].copy()
# DF with market and preiction data
market_df['Prob_Away'] = y_proba_opt[:, 0]
market_df['Prob_Draw'] = y_proba_opt[:, 1]
market_df['Prob_Home'] = y_proba_opt[:, 2]
# Calculating EV
market_df['EV_Away'] = (market_df['Prob_Away'] * market_df['B365A']) - 1
market_df['EV_Draw'] = (market_df['Prob_Draw'] * market_df['B365D']) - 1
market_df['EV_Home'] = (market_df['Prob_Home'] * market_df['B365H']) - 1

# Function to find matches with edge
def find_edge(row):
    edges = {'Away': row['EV_Away'], 'Draw': row['EV_Draw'], 'Home': row['EV_Home']}
    best_market = max(edges, key=edges.get)
    max_ev = edges[best_market]

    if max_ev > 0:
        return f"{best_market} (+{max_ev:.2f})"
    else:
        return "No Edge"
market_df['Signal'] = market_df.apply(find_edge, axis=1)

In [32]:
final_display = market_df[[
    'Date', 'HomeTeam', 'AwayTeam', 'FTR', 
    'B365H', 'Prob_Home', 'EV_Home', 
    'B365A', 'Prob_Away', 'EV_Away',
    'Signal'
]]


In [33]:
value_plays = final_display[final_display['Signal'] != "No Edge"].sort_values(by='Date')

print(f"Total Matches Analyzed: {len(market_df)}")
print(f"Value Plays Identified: {len(value_plays)}\n")
display(value_plays)

Total Matches Analyzed: 56
Value Plays Identified: 56



,Date,HomeTeam,AwayTeam,FTR,B365H,Prob_Home,EV_Home,B365A,Prob_Away,EV_Away,Signal
224,2024-04-20,Sheffield United,Burnley,A,2.75,0.285073,-0.216050,2.45,0.299961,-0.265096,Draw (+0.49)
225,2024-04-20,Wolves,Arsenal,A,8.50,0.162738,0.383277,1.40,0.638155,-0.106583,Home (+0.38)
226,2024-04-21,Crystal Palace,West Ham,H,2.10,0.199656,-0.580722,3.40,0.507857,0.726715,Away (+0.73)
227,2024-04-21,Everton,Nott'm Forest,H,2.05,0.305022,-0.374705,3.60,0.383530,0.380709,Away (+0.38)
228,2024-04-21,Aston Villa,Bournemouth,H,1.85,0.262032,-0.515241,3.60,0.433767,0.561561,Away (+0.56)
229,2024-04-21,Fulham,Liverpool,A,5.25,0.297376,0.561222,1.57,0.358464,-0.437211,Home (+0.56)
230,2024-04-23,Arsenal,Chelsea,H,1.44,0.404412,-0.417647,6.00,0.213763,0.282581,Draw (+0.91)
231,2024-04-24,Wolves,Bournemouth,A,2.70,0.148070,-0.600210,2.50,0.443423,0.108559,Draw (+0.47)
232,2024-04-24,Crystal Palace,Newcastle,H,2.75,0.269133,-0.259883,2.40,0.428500,0.028400,Draw (+0.09)
233,2024-04-24,Everton,Liverpool,H,7.00,0.164859,0.154010,1.40,0.606599,-0.150761,Home (+0.15)


## Despite 'edge' in many matches, result tends to not pan out. Reasons? Elementary model just too simple for large industrial-scale algorithms and analysis. Need much larger testing  set to allow for large number- greater chance of low-likelihood events to hit.

# To be Continued....

### *Assistance from Gemini AI was used